# Project 2 — Many-Time Pad: Analysis Starter

**Status:** workspace prepared; assignment dataset and required deliverables pending.

This notebook provides input validation and pairwise XOR inspection. It does not recover the key or solve the assignment yet. Read the project instructions before choosing an attack strategy.

Keep the official ciphertexts in `data/ciphertexts.txt`, one hexadecimal ciphertext per nonempty line, in their original order. Record the source and any separately designated target in `analysis_notes.md`. Do not substitute the OTP demonstration inputs for the assignment dataset.


## Why compare ciphertexts?

If two messages reuse the same key bytes at the same positions:

`C_i = M_i XOR K` and `C_j = M_j XOR K`

then:

`C_i XOR C_j = M_i XOR M_j`

The key cancels because `K XOR K = 0`. The result is the XOR of two plaintexts, not either plaintext by itself. Hexadecimal is a representation of the bytes; it does not change the XOR operation.

For messages of different lengths, compare only the overlapping prefix. This notebook assumes the ciphertexts begin at the same key offset; confirm that assumption against the exercise instructions.

The letter/space pattern in the existing hints can suggest candidates, but it does not prove a space or a recovered character. Record evidence before accepting a guess.


In [ ]:
from pathlib import Path
from itertools import combinations


def load_ciphertexts(path: Path) -> list[bytes]:
    """Read one hex ciphertext per nonempty line; retain the supplied order."""
    ciphertexts = []
    for line_number, line in enumerate(path.read_text(encoding="ascii").splitlines(), 1):
        hex_text = "".join(line.split())
        if not hex_text:
            continue
        try:
            ciphertext = bytes.fromhex(hex_text)
        except ValueError as exc:
            raise ValueError(
                f"{path.name}, line {line_number}: expected an even number of hexadecimal digits."
            ) from exc
        ciphertexts.append(ciphertext)
    if len(ciphertexts) < 2:
        raise ValueError("Provide at least two ciphertexts for pairwise comparison.")
    return ciphertexts


def xor_overlap(left: bytes, right: bytes) -> bytes:
    """XOR matching offsets over the shorter input; do not pad either message."""
    return bytes(a ^ b for a, b in zip(left, right))


def printable_preview(data: bytes) -> str:
    """Show printable ASCII bytes; use a dot for everything else."""
    return "".join(chr(value) if 32 <= value <= 126 else "." for value in data)


## Locate and inspect inputs

Run JupyterLab from `Many_Time_Pad/` or the repository root. The cell below accepts either working directory.

Input format: hexadecimal digits with optional whitespace, one complete ciphertext per line. Leave out labels, commas, quotation marks, and `0x` prefixes. If the official source uses another format, keep the original and document any conversion.

No dataset is bundled with this starter. A missing file produces a readiness message and allows the remaining cells to run.


In [ ]:
workspace_candidates = [Path("Many_Time_Pad"), Path(".")]
workspace = next(
    (folder for folder in workspace_candidates if (folder / "data").is_dir()),
    None,
)
if workspace is None:
    raise FileNotFoundError(
        "Run this notebook from the repository root or Many_Time_Pad/."
    )

DATA_FILE = workspace / "data" / "ciphertexts.txt"
ciphertexts = []

if not DATA_FILE.is_file():
    print(f"Ready for inputs: add the assignment ciphertexts to {DATA_FILE}.")
else:
    ciphertexts = load_ciphertexts(DATA_FILE)
    print("Label | Length in bytes")
    print("-----------------------")
    for index, ciphertext in enumerate(ciphertexts, 1):
        print(f"C{index:02d} | {len(ciphertext)}")


## Pairwise XOR inspection

Choose the one-based message numbers below. The preview replaces nonprintable bytes with dots; those dots are display markers, not recovered characters. Printable output is also not evidence of successful decryption.

The summary compares all message pairs. Use the selected-pair view when inspecting exact byte offsets.


In [ ]:
LEFT_MESSAGE = 1
RIGHT_MESSAGE = 2

if ciphertexts:
    if not (1 <= LEFT_MESSAGE <= len(ciphertexts) and 1 <= RIGHT_MESSAGE <= len(ciphertexts)):
        raise ValueError("Choose message numbers from the input inventory.")
    if LEFT_MESSAGE == RIGHT_MESSAGE:
        raise ValueError("Select two different messages.")

    pair_xor = xor_overlap(
        ciphertexts[LEFT_MESSAGE - 1],
        ciphertexts[RIGHT_MESSAGE - 1],
    )
    print(f"C{LEFT_MESSAGE:02d} XOR C{RIGHT_MESSAGE:02d}")
    print("Overlapping bytes:", len(pair_xor))
    print("Hex:", pair_xor.hex())
    print("Printable preview:", printable_preview(pair_xor))
    print("\nOffset | XOR byte | Printable preview")
    for offset, value in enumerate(pair_xor):
        print(f"{offset:6d} | {value:02x}       | {printable_preview(bytes([value]))}")
else:
    print("Pair inspection will be available once ciphertexts are supplied.")


In [ ]:
if ciphertexts:
    print("Pair | Overlapping bytes | Hex preview (first 16 bytes)")
    for (left_index, left), (right_index, right) in combinations(
        enumerate(ciphertexts, 1), 2
    ):
        result = xor_overlap(left, right)
        print(f"C{left_index:02d}/C{right_index:02d} | {len(result)} | {result[:16].hex()}")
else:
    print("Pair summary is waiting for the dataset.")


## Next steps

1. Confirm the assignment's dataset, plaintext character set, key-reuse assumptions, and required outputs.
2. Preserve ciphertext ordering and identify any designated target message.
3. Use pairwise XOR patterns to propose hypotheses; record message labels and zero-based byte offsets in `analysis_notes.md`.
4. Add space-candidate scoring or crib testing only after confirming the required method.
5. Cross-check any candidate key bytes against other ciphertexts at the same offsets.
6. Keep unknown characters explicit, and distinguish guesses from verified recovery.
7. Add final results and regenerate HTML/PDF exports when the analysis is ready for review.
